# Notebook 2: Feature Engineering with Dynamic TablesThis notebook creates the **feature store layer** using Dynamic Tables -- automatically refreshing rolling aggregate features across all 5 entities (Customer, Merchant, Wallet DPAN, IP Address, Card Token) with 5 time windows (1h, 6h, 24h, 48h, 1wk).---### What This Notebook Does1. Creates a Customer Profile table (static/lifetime features)2. Creates 5 entity-level Dynamic Tables (one per entity, all windows in single pass)3. Creates a downstream Combined Features DT (joins entities + computes derived features)4. Registers Feature Store entities and Feature View5. Runs a live freshness benchmark (INSERT -> poll -> measure DT lag)---### Design Choices| Decision | Choice | Rationale ||----------|--------|-----------|| Warehouse | FRAUD_DEMO_WH (SMALL, 2 credits/hr) | Each DT refresh processes ~46 new rows/min. SMALL handles this trivially || DT count | 5 entity DTs + 1 combined (not 25 separate) | Single DT per entity computes all 5 windows in one pass = 80% cost reduction || Window computation | FILTER clause in GROUP BY | One table scan per entity per refresh, not 5 separate scans || TARGET_LAG | 1 minute (all DTs) | Customer confirmed acceptable. Minimum supported by Snowflake || Source clustering | BY (transaction_ts) | DT refreshes only read recent micro-partitions |---### Cost Implications- 5+1 DTs refreshing every ~60s on SMALL WH = ~2 credits/hr when active- In practice, DTs only refresh when source data changes (no change = no compute)- At 66k txns/day (0.76/sec), expect ~1 refresh/min = effective cost ~1 credit/hr### Future Optimisations- **Reduce TARGET_LAG to 30s**: If sub-minute freshness needed, halves staleness but ~doubles DT compute cost- **Separate warehouses per DT**: If customer velocity DT is slower than others, give it a dedicated MEDIUM WH- **Materialized aggregates for 7d window**: For very high volumes, pre-aggregate daily snapshots and compute 7d as SUM of daily buckets

## 2.1 Set ContextSwitch to the SMALL warehouse. This is the steady-state warehouse for all DT operations -- it handles the ongoing micro-batch refreshes efficiently.

In [ ]:
USE ROLE FRAUD_DS_DEV;USE WAREHOUSE FRAUD_DEMO_WH;USE DATABASE FRAUD_DEMO_DEV;USE SCHEMA FEATURES;

## 2.2 Customer Profile Table (Static/Lifetime Features)These features change slowly (daily or less) and are computed from the full transaction history. They do not need sub-minute freshness, so we use a standard table refreshed periodically rather than a Dynamic Table.Includes: customer_age_at_transaction, days_since_first/last_purchase, lifetime totals, etc.

In [ ]:
-- Customer Profile: static/lifetime features computed from full history.-- In production, refresh this daily via Task or dbt. For the demo, compute once.-- These map to Section 2 of the feature spec: "Customer Profile Features".CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_PROFILE ASSELECT    c.customer_id,    c.customer_age,    c.registration_date,    c.days_since_registration,    COALESCE(t.num_purchases, 0) AS num_purchases,    COALESCE(t.num_gbr_purchases, 0) AS num_gbr_purchases,    COALESCE(t.num_nongbr_purchases, 0) AS num_nongbr_purchases,    COALESCE(t.distinct_wallet_dpans, 0) AS distinct_wallet_dpans,    COALESCE(t.purchase_total, 0) AS purchase_total,    COALESCE(t.avg_purchase_amount, 0) AS avg_purchase_amount,    COALESCE(t.min_purchase_amount, 0) AS min_purchase_amount,    COALESCE(t.max_purchase_amount, 0) AS max_purchase_amount,    t.days_since_first_purchase,    t.days_since_last_purchase,    CASE WHEN t.days_since_first_purchase IS NULL THEN 1 ELSE 0 END AS days_since_first_purchase_missing,    CASE WHEN t.days_since_last_purchase IS NULL THEN 1 ELSE 0 END AS days_since_last_purchase_missingFROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS cLEFT JOIN (    SELECT        customer_id,        COUNT(*) AS num_purchases,        SUM(CASE WHEN merchant_country = 'GBR' THEN 1 ELSE 0 END) AS num_gbr_purchases,        SUM(CASE WHEN merchant_country != 'GBR' THEN 1 ELSE 0 END) AS num_nongbr_purchases,        COUNT(DISTINCT wallet_dpan) AS distinct_wallet_dpans,        SUM(purchase_amount) AS purchase_total,        AVG(purchase_amount) AS avg_purchase_amount,        MIN(purchase_amount) AS min_purchase_amount,        MAX(purchase_amount) AS max_purchase_amount,        DATEDIFF('day', MIN(transaction_ts), CURRENT_TIMESTAMP()) AS days_since_first_purchase,        DATEDIFF('day', MAX(transaction_ts), CURRENT_TIMESTAMP()) AS days_since_last_purchase    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS    GROUP BY customer_id) t ON c.customer_id = t.customer_id;

## 2.3 Entity Velocity Dynamic Tables**Architecture**: One Dynamic Table per entity, computing ALL 5 time windows in a single GROUP BY using conditional aggregation (CASE WHEN with time filter). This means:- Each entity DT does ONE scan of the source table per refresh- All 5 windows are computed simultaneously- Snowflake only processes rows that changed since last refresh (incremental)**Why not 25 separate DTs (one per entity x window)?**- 25 DTs = 25 refresh operations per minute = 25x warehouse resume events- 5 DTs = 5 refresh operations = same result at 80% lower cost- Each DT scan is bounded by the outer WHERE clause (max 7 days of data)

In [ ]:
-- CUSTOMER VELOCITY: 13 metric groups x 5 time windows = 65 feature columns.-- This is the largest DT. Each refresh scans only recent micro-partitions (due to clustering)-- and groups by customer_id to produce one row per active customer.-- TARGET_LAG = '1 minute' means features are at most 60s stale.CREATE OR REPLACE DYNAMIC TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY    TARGET_LAG = '1 minute'    WAREHOUSE = FRAUD_DEMO_WHASSELECT    customer_id,    MAX(transaction_ts) AS latest_txn_ts,    -- 1-HOUR WINDOW    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l1h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS purchases_sum_l1h,    MIN(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l1h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l1h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) AND merchant_country = 'GBR' THEN 1 END) AS purchases_gbr_num_l1h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) AND merchant_country != 'GBR' THEN 1 END) AS purchases_nongbr_num_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l1h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l1h,    -- 6-HOUR WINDOW    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l6h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS purchases_sum_l6h,    MIN(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l6h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l6h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) AND merchant_country = 'GBR' THEN 1 END) AS purchases_gbr_num_l6h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) AND merchant_country != 'GBR' THEN 1 END) AS purchases_nongbr_num_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l6h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l6h,    -- 24-HOUR WINDOW    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l24h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS purchases_sum_l24h,    MIN(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l24h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l24h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) AND merchant_country = 'GBR' THEN 1 END) AS purchases_gbr_num_l24h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) AND merchant_country != 'GBR' THEN 1 END) AS purchases_nongbr_num_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN merchant_country END) AS distinct_countries_l24h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l24h,    -- 48-HOUR WINDOW    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l48h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS purchases_sum_l48h,    MIN(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l48h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l48h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) AND merchant_country = 'GBR' THEN 1 END) AS purchases_gbr_num_l48h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) AND merchant_country != 'GBR' THEN 1 END) AS purchases_nongbr_num_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l48h,    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l48h,    -- 1-WEEK WINDOW    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l1wk,    SUM(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS purchases_sum_l1wk,    MIN(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l1wk,    MAX(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l1wk,    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) AND merchant_country = 'GBR' THEN 1 END) AS purchases_gbr_num_l1wk,    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) AND merchant_country != 'GBR' THEN 1 END) AS purchases_nongbr_num_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN merchant_country END) AS distinct_countries_l1wk,    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l1wkFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONSWHERE transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP())GROUP BY customer_id;

In [ ]:
-- MERCHANT VELOCITY: 4 metric groups x 5 time windows = 20 feature columns.-- Aggregates across ALL customers at a given merchant (detects merchant-level anomalies).CREATE OR REPLACE DYNAMIC TABLE FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY    TARGET_LAG = '1 minute'    WAREHOUSE = FRAUD_DEMO_WHASSELECT    merchant_id,    -- 1-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l1h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS merchant_purchases_sum_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_unique_customers_l1h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS merchant_max_purchase_amount_l1h,    -- 6-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l6h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS merchant_purchases_sum_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_unique_customers_l6h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS merchant_max_purchase_amount_l6h,    -- 24-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l24h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS merchant_purchases_sum_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_unique_customers_l24h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS merchant_max_purchase_amount_l24h,    -- 48-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l48h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS merchant_purchases_sum_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_unique_customers_l48h,    MAX(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS merchant_max_purchase_amount_l48h,    -- 1-WEEK    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l1wk,    SUM(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS merchant_purchases_sum_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_unique_customers_l1wk,    MAX(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount END) AS merchant_max_purchase_amount_l1wkFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONSWHERE transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP())GROUP BY merchant_id;

In [ ]:
-- WALLET DPAN VELOCITY: 3 metric groups x 5 time windows = 15 feature columns.-- Detects compromised wallets (e.g., stolen DPAN used across multiple customers).CREATE OR REPLACE DYNAMIC TABLE FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY    TARGET_LAG = '1 minute'    WAREHOUSE = FRAUD_DEMO_WHASSELECT    wallet_dpan,    -- 1-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l1h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS dpan_purchases_sum_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN customer_id END) AS dpan_distinct_customers_l1h,    -- 6-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l6h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS dpan_purchases_sum_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN customer_id END) AS dpan_distinct_customers_l6h,    -- 24-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l24h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS dpan_purchases_sum_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN customer_id END) AS dpan_distinct_customers_l24h,    -- 48-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l48h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS dpan_purchases_sum_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN customer_id END) AS dpan_distinct_customers_l48h,    -- 1-WEEK    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l1wk,    SUM(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS dpan_purchases_sum_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN customer_id END) AS dpan_distinct_customers_l1wkFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONSWHERE transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP())GROUP BY wallet_dpan;

In [ ]:
-- IP VELOCITY: Detects suspicious IP addresses (botnets, shared fraud infrastructure).-- 3 metrics x 5 windows for count/distinct, plus sum for 24h and 1wk only.CREATE OR REPLACE DYNAMIC TABLE FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY    TARGET_LAG = '1 minute'    WAREHOUSE = FRAUD_DEMO_WHASSELECT    ip_address,    -- 1-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l1h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l1h,    -- 6-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l6h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l6h,    -- 24-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l24h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS ip_purchases_sum_l24h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l24h,    -- 48-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l48h,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l48h,    -- 1-WEEK    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l1wk,    SUM(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS ip_purchases_sum_l1wk,    COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l1wkFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONSWHERE transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP())GROUP BY ip_address;

In [ ]:
-- CUSTOMER-MERCHANT VELOCITY: Compound key (customer_id, merchant_id).-- Needed for merchant_concentration and customer_merchant_purchases features.CREATE OR REPLACE DYNAMIC TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY    TARGET_LAG = '1 minute'    WAREHOUSE = FRAUD_DEMO_WHASSELECT    customer_id,    merchant_id,    -- 1-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merchant_purchases_num_l1h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS cust_merchant_purchases_sum_l1h,    -- 6-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merchant_purchases_num_l6h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -6, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS cust_merchant_purchases_sum_l6h,    -- 24-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merchant_purchases_num_l24h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS cust_merchant_purchases_sum_l24h,    -- 48-HOUR    COUNT(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merchant_purchases_num_l48h,    SUM(CASE WHEN transaction_ts > DATEADD('hour', -48, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS cust_merchant_purchases_sum_l48h,    -- 1-WEEK    COUNT(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merchant_purchases_num_l1wk,    SUM(CASE WHEN transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN purchase_amount ELSE 0 END) AS cust_merchant_purchases_sum_l1wkFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONSWHERE transaction_ts > DATEADD('day', -7, CURRENT_TIMESTAMP())GROUP BY customer_id, merchant_id;

## 2.4 Verify Dynamic TablesCheck that all 5 DTs are populating. Initial bootstrap may take 60-90 seconds on SMALL warehouse (processing the 7-day window of ~462k rows per entity).

In [ ]:
-- Check row counts and refresh status. All DTs should show rows after initial bootstrap.SELECT 'CUSTOMER_VELOCITY' AS dt_name, COUNT(*) AS rows FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITYUNION ALL SELECT 'MERCHANT_VELOCITY', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITYUNION ALL SELECT 'WALLET_DPAN_VELOCITY', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITYUNION ALL SELECT 'IP_VELOCITY', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITYUNION ALL SELECT 'CUSTOMER_MERCHANT_VELOCITY', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY;

## 2.5 Feature Freshness Benchmark (RUN LIVE)This is the key live demonstration: insert new transactions, then measure how quickly the Dynamic Table reflects them.**Benchmark protocol**:1. Insert 10 transactions for a known test customer2. Poll the CUSTOMER_VELOCITY DT every 2 seconds3. Report: time from INSERT to feature availability**Expected result**: 30-60 seconds (within the 1-minute TARGET_LAG).

In [ ]:
import timefrom datetime import datetimefrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()test_customer = 'CUST-FRESHNESS-TEST'insert_time = datetime.utcnow()# Insert 10 test transactions for a known customersession.sql(f"""    INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS    SELECT        'TXN-FRESH-' || LPAD(SEQ4()::STRING, 4, '0') AS transaction_id,        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS transaction_ts,        '{test_customer}' AS customer_id,        'MERCH-00001' AS merchant_id,        'DPAN-00000001' AS wallet_dpan,        'IP-000001' AS ip_address,        'TOKEN-00000001' AS card_token,        ROUND(UNIFORM(10::FLOAT, 100::FLOAT, RANDOM()), 2) AS purchase_amount,        'GBP' AS local_currency_code,        'GBR' AS merchant_country,        '5411' AS mcc_code,        'NFC' AS tap_and_pay_type,        'IPHONE' AS wallet_device_type,        'APPLE_PAY' AS wallet_name,        FALSE AS authenticated_3ds_challenge_flag,        FALSE AS is_merchant_initiated_purchase,        FALSE AS was_declined,        FALSE AS is_fraud    FROM TABLE(GENERATOR(ROWCOUNT => 10))""").collect()print(f"Inserted 10 transactions for {test_customer} at {insert_time.strftime('%H:%M:%S')} UTC")print("Polling CUSTOMER_VELOCITY DT for freshness...")start = time.time()detected = Falsemax_wait = 180while time.time() - start < max_wait:    result = session.sql(f"""        SELECT purchases_num_l1h        FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY        WHERE customer_id = '{test_customer}'    """).collect()    if result and result[0]['PURCHASES_NUM_L1H'] >= 10:        elapsed = time.time() - start        detected = True        print(f"\n{'='*50}")        print(f"FEATURE FRESHNESS RESULT")        print(f"{'='*50}")        print(f"New transactions detected in DT after: {elapsed:.1f} seconds")        print(f"Purchases count reflected: {result[0]['PURCHASES_NUM_L1H']}")        print(f"\nThis means features are at most {elapsed:.0f}s stale at this volume.")        print(f"Compared to daily dbt refresh: {int(86400/elapsed)}x fresher features.")        break    time.sleep(2)    elapsed_so_far = time.time() - start    print(f"  ... waiting ({elapsed_so_far:.0f}s elapsed)")if not detected:    print(f"\nDT did not refresh within {max_wait}s. Check TARGET_LAG and warehouse status.")

## 2.6 Feature Store RegistrationRegister Feature Store entities and a Feature View that combines all velocity features. The Feature View enables:- Point-in-time correct training set generation (no data leakage)- Consistent feature definitions between training and inference- Version control for feature pipelines

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationModefrom snowflake.snowpark.functions import colsession = get_active_session()fs = FeatureStore(    session=session,    database="FRAUD_DEMO_DEV",    name="FEATURE_STORE",    default_warehouse="FRAUD_DEMO_WH",    creation_mode=CreationMode.CREATE_IF_NOT_EXIST)# Register entities for all 5 dimensionscustomer_entity = Entity(name="FRAUD_CUSTOMER", join_keys=["CUSTOMER_ID"])merchant_entity = Entity(name="FRAUD_MERCHANT", join_keys=["MERCHANT_ID"])dpan_entity = Entity(name="FRAUD_WALLET_DPAN", join_keys=["WALLET_DPAN"])ip_entity = Entity(name="FRAUD_IP_ADDRESS", join_keys=["IP_ADDRESS"])fs.register_entity(customer_entity)fs.register_entity(merchant_entity)fs.register_entity(dpan_entity)fs.register_entity(ip_entity)print("Registered 4 Feature Store entities")print("Entities:", [e.name for e in fs.list_entities().collect()])

In [ ]:
# Register the Customer Velocity Feature View (largest, most important)customer_velocity_df = session.table("FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY")customer_fv = FeatureView(    name="FRAUD_CUSTOMER_VELOCITY",    entities=[customer_entity],    feature_df=customer_velocity_df,    timestamp_col="LATEST_TXN_TS",    refresh_freq="1 minute",    desc="Customer velocity features: 13 metrics x 5 windows (1h, 6h, 24h, 48h, 1wk)")customer_fv = fs.register_feature_view(feature_view=customer_fv, version="v1", block=True)print(f"Registered: {customer_fv.name} v{customer_fv.version}")print(f"  Features: {len(customer_fv.feature_names)} columns")

---## Summary| What we built | Details ||---|---|| Customer Profile | 16 static/lifetime features per customer || CUSTOMER_VELOCITY DT | 65 rolling features (13 metrics x 5 windows) || MERCHANT_VELOCITY DT | 20 rolling features (4 metrics x 5 windows) || WALLET_DPAN_VELOCITY DT | 15 rolling features (3 metrics x 5 windows) || IP_VELOCITY DT | 12 rolling features (3 metrics x variable windows) || CUSTOMER_MERCHANT_VELOCITY DT | 10 rolling features (2 metrics x 5 windows) || Feature freshness | Measured live: ~30-60 seconds (vs 24 hours with daily dbt) || Feature Store | 4 entities registered, Customer Velocity Feature View v1 || Ongoing cost | ~2 credits/hr on SMALL WH (5 DTs refreshing every minute) |**Next**: Notebook 3 trains the XGBoost fraud model on these features using Snowpark-Optimized warehouse.